In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from keras import layers , Sequential

In [2]:

url = "https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt"
path = tf.keras.utils.get_file('shakespeare.txt', url)

text = open(path, 'rb').read().decode(encoding='utf-8')
print(f"Total characters: {len(text)}")
print(text[:50])

Total characters: 1115394
First Citizen:
Before we proceed any further, hear


In [3]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer()

tokenizer.fit_on_texts([text])

total_words = len(tokenizer.word_index) + 1

print(f"Total unique word (Vocabulary size) : {total_words}")

Total unique word (Vocabulary size) : 12633


In [4]:
print(f"Index of word 'the' : {tokenizer.word_index.get('the')}" )
print(f"Index of word 'king' : {tokenizer.word_index.get('king')}")

Index of word 'the' : 1
Index of word 'king' : 34


In [5]:
input_sequences = []

# Text ko lines me todo (Shakespeare format me har line alag hai)
for line in text.split('\n'):
    if line.strip() == "":  # khaali lines skip karo
        continue
    token_list = tokenizer.texts_to_sequences([line])[0]  # line ko numbers me convert karo
    
    # Sliding window — progressively badi sequences banao
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

print(f"Total training sequences: {len(input_sequences)}")
print("Example sequence:", input_sequences[5])

Total training sequences: 171312
Example sequence: [139, 35, 969, 143, 668, 127]


In [6]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_sequence_len = max([len(seq) for seq in input_sequences])
print(f"Max sequence length : {max_sequence_len}")

input_sequence = np.array(
    pad_sequences(
        input_sequences , maxlen = max_sequence_len , padding = 'pre'
    )
)
print("Padded example:", input_sequences[5])

Max sequence length : 16
Padded example: [139, 35, 969, 143, 668, 127]


In [7]:
X = input_sequence[ : , : -1]
y = input_sequence[ : ,   -1]

print(f"X shape : {X.shape}")
print(f"y shape : {y.shape}")

X shape : (171312, 15)
y shape : (171312,)


In [8]:
from tensorflow.keras.utils import to_categorical
y = to_categorical(y , num_classes = total_words)

print(f"y shape after one-hot : {y.shape}")

y shape after one-hot : (171312, 12633)


In [9]:
model = Sequential([
    layers.Input(shape=(max_sequence_len , )),
    layers.Embedding(input_dim = total_words , output_dim = 100),
    layers.GRU(150), # size of cell state and hidden state is 150
    layers.Dense(total_words , activation = "softmax")
])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 16, 100)        │     1,263,300 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 150)            │       113,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 12633)          │     1,907,583 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,284,283 (12.53 MB)

 Trainable params: 3,284,283 (12.53 MB)

 Non-trainable params: 0 (0.00 B)

In [10]:
model.compile(
    optimizer = 'adam',
    loss = "categorical_crossentropy",
    metrics = ["accuracy"]
)

In [ ]:
history = model.fit(
    X , y,
    epochs = 50,
    batch_size = 64,
    metrics = ["accuracy"],
    verbose = 1
)